# GRPO RL — train & eval

Drives the GRPO trainer and the standalone eval that live next to this notebook
(`train_grpo.py`, `eval.py`). Designed for a **Colab A100** runtime, and bootstraps
from an empty `/content` (clone → install → run).

**The reward is the eval scorer** — both call `shared/scoring.py::score_one`, so a
reward of `1.0` means a completion that would be marked correct at eval. No leakage:
we train on `public.jsonl[100:]` and evaluate on the first-100 `default` slice.

## Where the trained model is stored

GRPO trains a **LoRA adapter** on top of base `Qwen/Qwen3-4B-Thinking-2507` (not a
full model). For a run named `<run_name>`:

| Artifact | Path |
|---|---|
| Final adapter (load this) | `experiments/reinforcement_learning/results/<run_name>/final_adapter/` |
| Intermediate checkpoints | `experiments/reinforcement_learning/results/<run_name>/checkpoint-<step>/` |
| Resolved train config | `experiments/reinforcement_learning/results/<run_name>/train_config.yaml` |
| Eval results (JSONL) | `experiments/reinforcement_learning/results/<run_name>.jsonl` |

The `final_adapter/` dir holds `adapter_config.json` + `adapter_model.safetensors`
+ tokenizer files — exactly what the shared eval runner loads.

> **Colab note:** `/content` is wiped when the runtime resets, so the cloned repo
> and any trained adapters are ephemeral. To keep an adapter, copy
> `results/<run_name>/final_adapter/` to Google Drive (mount it and `cp -r`).

## 0. Bootstrap: clone + install

On a fresh Colab there's nothing checked out, so we clone the (public) repo, `cd`
into it, and export `REPO_ROOT` (the Hydra searchpath needs it; Colab's cwd is
`/content`). Idempotent — it won't re-clone if the directory already exists, and it
reuses an existing checkout if you're running inside one.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/DarinAnthony/151B_SP26_Competition.git'
BRANCH = 'main'
CLONE_DIR = Path('/content/151B_SP26_Competition')

def find_repo_root(start: Path):
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'shared').is_dir():
            return p
    return None

# Reuse an existing checkout if we're already inside one (e.g. running locally).
REPO_ROOT = find_repo_root(Path.cwd())
if REPO_ROOT is None:
    if not CLONE_DIR.exists():
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)], check=True)
    REPO_ROOT = CLONE_DIR

os.chdir(REPO_ROOT)
os.environ['REPO_ROOT'] = str(REPO_ROOT)  # inherited by the ! shell cells below
print('REPO_ROOT =', REPO_ROOT)

In [ ]:
# Install the project + RL extra (adds TRL). pyproject pins transformers 4.x / vllm
# 0.10 (the comments there explain why: transformers 5.x breaks vLLM's Qwen3
# tokenizer load). On Colab this downgrades the preinstalled torch/transformers, so
# **restart the runtime after this cell** (Runtime > Restart), then run from cell 0
# again — the clone is cached so it won't re-download.
!pip -q install -e ".[rl]"
import peft, transformers, trl, vllm
print('trl', trl.__version__, '| transformers', transformers.__version__, '| vllm', vllm.__version__, '| peft', peft.__version__)

In [ ]:
# Run label. Adapter + eval JSONL are named after this.
RUN_NAME = 'grpo_demo_v1'
RESULTS_DIR = REPO_ROOT / 'experiments/reinforcement_learning/results'
ADAPTER_DIR = RESULTS_DIR / RUN_NAME / 'final_adapter'
print('adapter will be saved to:', ADAPTER_DIR)

## 1. Baseline eval (base model, no adapter)

Get the pre-RL number on the first-100 slice so the trained run has something to
beat. `adapter_path` is left null → base Qwen.

In [ ]:
!python -m experiments.reinforcement_learning.src.eval run_name=grpo_base_eval

## 2. Smoke train (optional plumbing check)

2 steps, 8 rows, HF rollout (no vLLM engine to spin up). Just confirms the loop
runs and the adapter saves — not for real signal. Skip once you trust the setup.

In [ ]:
!python -m experiments.reinforcement_learning.src.train_grpo grpo=smoke run_name=grpo_smoke

## 3. Demo train (GRPO)

50 steps, group size 4, vLLM colocate rollouts (auto-falls back to HF if vLLM is
unavailable). Override anything on the CLI, e.g. add
`grpo.max_steps=200 grpo.num_generations=8`, or warm-start from an SFT adapter with
`model.init_adapter_path=<dir>`.

In [ ]:
!python -m experiments.reinforcement_learning.src.train_grpo grpo=demo run_name={RUN_NAME}

## 4. Eval the trained adapter

Same first-100 slice, greedy decoding, now with the trained LoRA adapter loaded.

In [ ]:
!python -m experiments.reinforcement_learning.src.eval adapter_path={ADAPTER_DIR} run_name={RUN_NAME}_eval

## 5. Inspect results

Load the eval JSONL, print the overall / MCQ / free-form accuracy, and peek at a few
rows (gold vs. extracted answer).

In [ ]:
import json
import pandas as pd

def summarize(run_label):
    path = RESULTS_DIR / f'{run_label}.jsonl'
    rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    df = pd.DataFrame(rows)
    mcq, free = df[df.is_mcq], df[~df.is_mcq]
    print(f'{run_label:20s}  overall {df.correct.mean():.1%}  |  MCQ {mcq.correct.mean():.1%}  |  free {free.correct.mean():.1%}  (n={len(df)})')
    return df

base_df = summarize('grpo_base_eval')
trained_df = summarize(f'{RUN_NAME}_eval')
trained_df[['id', 'is_mcq', 'gold', 'extracted', 'correct', 'n_response_tokens']].head(10)